# Notebook 11 — Protocolo Experimental con Participantes Humanos (Fases 1-4)

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  
**Fase:** 5 — Validación con participantes humanos

---

## Estructura del protocolo experimental

| Fase | Instrumento | Medio | Duración estimada |
|------|-------------|-------|-------------------|
| 1 | Consentimiento informado + firma electrónica | Este notebook | 5 min |
| 2 | Instrumento AdHoc (datos sociodemográficos y salud) | Este notebook | 10 min |
| 3 | BCSE — Guía de administración + registro de puntuaciones | Este notebook | 10 min |
| 4 | Prueba de salud visual | Este notebook | 10 min |
| 5 | 190 tareas de percepción de profundidad | **App nativa Meta Quest 2** | 30 min |

> **Nota sobre la Fase 5:** Las tareas de percepción de profundidad se administran mediante una aplicación nativa desarrollada en Unity para el dispositivo Meta Quest 2. Esta app presenta los estímulos en formato estereoscópico (imagen izquierda/derecha para KITTI; imagen duplicada para ilusiones), registra las respuestas del participante mediante los controles del dispositivo y guarda los datos directamente en Google Drive. El código fuente de la app se encuentra en `app_quest2/` del repositorio GitHub del proyecto.

## Instrucciones para el investigador

- Ejecutar las **Celdas 1 y 2** antes de que el participante vea la pantalla.
- Usar **Ver → Ocultar código** (Ctrl+M+L) para mostrar solo la interfaz al participante.
- Para sesiones piloto usar `PILOT01`, `PILOT02` como ID. Excluidos del análisis estadístico.
- Al completar la Fase 4, ejecutar la **Celda 7** para guardar todos los datos antes de pasar a la app Quest 2.

## Confidencialidad y protección de datos

Los datos se almacenan únicamente con ID anonimizado. El mapeo nombre↔ID se guarda cifrado en `data/private/`, separado de los datos de respuesta. Tratamiento conforme a Ley 1581 de 2012 (Habeas Data, Colombia).

## Ground truth validado

El ground truth de las tareas fue validado empíricamente: generado algorítmicamente (NB08-09), corregido para el dataset de ilusiones visuales, y verificado manualmente imagen por imagen por el investigador principal (15/15 imágenes confirmadas correctas).

---

## Celda 1 — Configuración inicial (solo investigador)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, datetime, hashlib
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

BASE_DIR    = '/content/drive/MyDrive/cognitive-depth-model'
DATA_DIR    = os.path.join(BASE_DIR, 'data', 'participants')
PRIVATE_DIR = os.path.join(BASE_DIR, 'data', 'private')
RESULTS_DIR = os.path.join(BASE_DIR, 'results', 'human_responses')
for d in [DATA_DIR, PRIVATE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('✓ Drive montado y rutas configuradas.')
print()
print('Sesiones registradas:')
existing = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('_session.json')])
pilots   = [e for e in existing if 'PILOT' in e]
reales   = [e for e in existing if 'PILOT' not in e]
print(f'  Reales: {len(reales)} | Pilotos: {len(pilots)}')
for e in existing:
    print(f'    {e}')

## Celda 2 — Registro del participante (solo investigador)

In [ ]:
# ── SOLO INVESTIGADOR ─────────────────────────────────────────
# Participantes reales: P001, P002, P003
# Sesiones piloto:      PILOT01, PILOT02

PARTICIPANT_ID = 'P001'   # ← CAMBIAR PARA CADA SESIÓN

FECHA_SESION = datetime.datetime.now().strftime('%Y-%m-%d')
HORA_INICIO  = datetime.datetime.now().strftime('%H:%M:%S')
SESSION_CODE = hashlib.md5(
    f'{PARTICIPANT_ID}{FECHA_SESION}{HORA_INICIO}'.encode()
).hexdigest()[:8].upper()

id_file = os.path.join(DATA_DIR, f'{PARTICIPANT_ID}_session.json')
if os.path.exists(id_file):
    print(f'⚠ Ya existe sesión para {PARTICIPANT_ID}. Verifica antes de continuar.')
else:
    es_piloto = 'PILOT' in PARTICIPANT_ID
    print(f'✓ ID: {PARTICIPANT_ID}  {"[PILOTO — excluido del análisis]" if es_piloto else "[PARTICIPANTE REAL]"}')
    print(f'  Fecha: {FECHA_SESION} | Hora: {HORA_INICIO} | Código: {SESSION_CODE}')
    print()
    print('Puedes compartir pantalla y ejecutar la Celda 3 → Consentimiento.')

session_data = {
    'participant_id': PARTICIPANT_ID,
    'es_piloto':      'PILOT' in PARTICIPANT_ID,
    'session_code':   SESSION_CODE,
    'fecha':          FECHA_SESION,
    'hora_inicio':    HORA_INICIO,
    'consentimiento': {},
    'adhoc':          {},
    'bcse':           {},
    'salud_visual':   {},
    'tareas_app':     'pendiente_app_quest2'
}

---
## FASE 1 — Consentimiento Informado
### Celda 3 — Presentación y firma electrónica

In [ ]:
display(HTML(f'''
<div style="font-family:Arial,sans-serif;max-width:820px;margin:0 auto;padding:20px;">
<div style="background:#1a5276;color:white;padding:15px;border-radius:8px 8px 0 0;text-align:center;">
  <h2 style="margin:0;">CONSENTIMIENTO INFORMADO</h2>
  <p style="margin:5px 0 0;">Grupo de Investigación: Sistemas Cognitivos Artificiales</p>
</div>
<div style="border:2px solid #1a5276;border-top:none;padding:25px;border-radius:0 0 8px 8px;background:#fdfefe;">
<p style="text-align:right;color:#555;">Manizales, {FECHA_SESION}</p>
<p><strong>Investigación:</strong> Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad</p>
<p><strong>Investigador principal:</strong> Jesús Goenaga Peña — Doctorando en Ciencias Cognitivas, Universidad Autónoma de Manizales</p>
<p><strong>Director:</strong> Luis Fernando Castillo Ossa (PhD)</p>
<h3>¿En qué consiste su participación?</h3>
<p>Si acepta participar, completará las siguientes actividades en una sesión de aproximadamente <strong>60 minutos</strong>:</p>
<ol>
  <li>Diligenciar un formulario con datos sociodemográficos y auto-reporte de salud (Instrumento AdHoc).</li>
  <li>Participar en la aplicación del Test Breve de Evaluación del Estado Cognitivo (BCSE).</li>
  <li>Realizar pruebas breves de salud visual (agudeza visual y visión binocular).</li>
  <li>Completar 190 tareas de percepción de profundidad visual mediante un dispositivo de realidad virtual (Meta Quest 2), indicando si un objeto marcado (A) aparece más cercano o más lejano que otro (B) en cada imagen.</li>
</ol>
<h3>Información importante</h3>
<ul>
  <li>Su participación es completamente <strong>libre y voluntaria</strong>. Puede retirarse en cualquier momento sin penalización.</li>
  <li>Se registrarán sus respuestas y tiempos de reacción. <strong>No</strong> se realizará registro fotográfico ni de video de su persona.</li>
  <li>Sus datos se almacenarán con un <strong>código anónimo</strong>. Su nombre nunca aparecerá en los archivos de datos.</li>
  <li>La información se usará exclusivamente para investigación científica y podrá publicarse de forma anonimizada.</li>
  <li>Recibirá una <strong>compensación económica razonable</strong> por su tiempo.</li>
  <li>No existen riesgos físicos conocidos. Si experimenta malestar visual o mareo con el dispositivo VR, puede solicitar pausa o detener la sesión.</li>
  <li>Contacto: jesus.goenagap@autonoma.edu.co</li>
</ul>
<h3>Confidencialidad y protección de datos</h3>
<p>Sus datos serán tratados conforme a la Ley 1581 de 2012 (Habeas Data, Colombia). Se almacenan en servidores protegidos con acceso restringido al equipo investigador. El mapeo entre su nombre y su código de participante se guarda cifrado y separado de los datos de respuesta.</p>
<div style="background:#eaf2ff;padding:12px;border-radius:6px;border-left:4px solid #1a5276;margin-top:18px;">
  <strong>Código de sesión:</strong> {SESSION_CODE}<br>
  <small>Conserve este código si desea solicitar la eliminación de sus datos en el futuro.</small>
</div>
</div></div>
'''))

print('─'*60)
print('FIRMA ELECTRÓNICA')
print('─'*60)
print()

nombre_w = widgets.Text(description='Nombre completo:',
    layout=widgets.Layout(width='500px'), style={'description_width':'160px'})
doc_w    = widgets.Text(description='No. documento:',
    layout=widgets.Layout(width='400px'), style={'description_width':'160px'})
acepta_w = widgets.Checkbox(value=False,
    description='He leído y comprendido el consentimiento y acepto participar voluntariamente.',
    layout=widgets.Layout(width='720px'))
btn_firma = widgets.Button(description='Confirmar firma electrónica',
    button_style='success', layout=widgets.Layout(width='280px', height='42px'))
out_firma = widgets.Output()

def on_firma(b):
    with out_firma:
        clear_output()
        if not acepta_w.value:
            print('⚠ Debe marcar la casilla de aceptación.')
            return
        if not nombre_w.value.strip() or not doc_w.value.strip():
            print('⚠ Complete todos los campos.')
            return
        ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        session_data['consentimiento'] = {
            'nombre_hash':  hashlib.sha256(nombre_w.value.strip().encode()).hexdigest(),
            'doc_hash':     hashlib.sha256(doc_w.value.strip().encode()).hexdigest(),
            'acepto':       True,
            'timestamp':    ts,
            'session_code': SESSION_CODE
        }
        with open(os.path.join(PRIVATE_DIR, f'{PARTICIPANT_ID}_identity.json'), 'w') as f:
            json.dump({
                'participant_id': PARTICIPANT_ID,
                'nombre':         nombre_w.value.strip(),
                'documento':      doc_w.value.strip(),
                'timestamp':      ts,
                'session_code':   SESSION_CODE
            }, f, ensure_ascii=False, indent=2)
        print(f'✓ Consentimiento firmado electrónicamente.')
        print(f'  Nombre: {nombre_w.value.strip()}')
        print(f'  Timestamp: {ts} | Código: {SESSION_CODE}')
        print()
        print('Ejecute la Celda 4 → Instrumento AdHoc.')

btn_firma.on_click(on_firma)
display(nombre_w, doc_w, acepta_w, btn_firma, out_firma)

---
## FASE 2 — Instrumento AdHoc
### Celda 4 — Datos sociodemográficos y salud

In [ ]:
display(HTML('<h2 style="font-family:Arial;color:#1a5276;border-bottom:2px solid #1a5276;padding-bottom:8px;">📋 INSTRUMENTO AdHoc</h2><p style="font-family:Arial;">Complete los siguientes campos. Sus respuestas son confidenciales.</p>'))

edad_w   = widgets.BoundedIntText(description='Edad:', min=18, max=99, value=30,
    layout=widgets.Layout(width='250px'), style={'description_width':'120px'})
genero_w = widgets.RadioButtons(description='Género:',
    options=['Masculino','Femenino','Otro'],
    layout=widgets.Layout(width='350px'), style={'description_width':'120px'})
educ_w   = widgets.Select(description='Nivel educativo:',
    options=['Sin educación formal','Primaria','Bachillerato','Técnico/Tecnológico',
             'Carrera profesional','Especialización','Maestría','Doctorado'],
    layout=widgets.Layout(width='450px'), style={'description_width':'150px'})
ocup_w   = widgets.Select(description='Ocupación:',
    options=['Sin empleo','Estudiante','Trabajador independiente','Trabajador dependiente','Otro'],
    layout=widgets.Layout(width='420px'), style={'description_width':'150px'})
civil_w  = widgets.Select(description='Estado civil:',
    options=['Soltero/a','Casado/a','Conviviente','Separado/a','Divorciado/a','Viudo/a'],
    layout=widgets.Layout(width='350px'), style={'description_width':'150px'})

display(HTML('<h3 style="font-family:Arial;">Datos sociodemográficos</h3>'))
display(edad_w, genero_w, educ_w, ocup_w, civil_w)

prob_vis_w = widgets.RadioButtons(description='¿Problema visual diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'270px'})
tipo_vis_w = widgets.SelectMultiple(description='Tipo:',
    options=['Miopía','Hipermetropía','Astigmatismo','Presbicia','Otro'],
    layout=widgets.Layout(width='380px'), style={'description_width':'80px'})
correc_w   = widgets.RadioButtons(description='Corrección visual:',
    options=['Gafas','Lentes de contacto','Ninguna'],
    layout=widgets.Layout(width='350px'), style={'description_width':'160px'})
examen_w   = widgets.RadioButtons(description='Frecuencia exámenes:',
    options=['Regularmente (1-2 años)','De vez en cuando (>2 años)','Nunca'],
    layout=widgets.Layout(width='450px'), style={'description_width':'190px'})

display(HTML('<h3 style="font-family:Arial;">Salud visual</h3>'))
display(prob_vis_w, tipo_vis_w, correc_w, examen_w)

t_cog_w   = widgets.RadioButtons(description='¿Trastorno neurocognitivo diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='450px'), style={'description_width':'310px'})
cambios_w = widgets.SelectMultiple(description='Cambios recientes:',
    options=['Cambios en memoria reciente','Dificultad para concentrarse',
             'Dificultad para resolver problemas','Ninguno'],
    layout=widgets.Layout(width='500px'), style={'description_width':'160px'})

display(HTML('<h3 style="font-family:Arial;">Salud neurocognitiva</h3>'))
display(t_cog_w, cambios_w)

t_psiq_w = widgets.RadioButtons(description='¿Trastorno psiquiátrico diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='450px'), style={'description_width':'310px'})
sint_w   = widgets.RadioButtons(description='¿Síntomas psiquiátricos recientes?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'260px'})
trat_w   = widgets.RadioButtons(description='¿Tratamiento psiquiátrico último año?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'270px'})

display(HTML('<h3 style="font-family:Arial;">Salud psiquiátrica</h3>'))
display(t_psiq_w, sint_w, trat_w)

btn_adhoc = widgets.Button(description='Guardar AdHoc',
    button_style='primary', layout=widgets.Layout(width='220px', height='42px'))
out_adhoc = widgets.Output()

def on_adhoc(b):
    with out_adhoc:
        clear_output()
        session_data['adhoc'] = {
            'edad': edad_w.value, 'genero': genero_w.value,
            'nivel_educativo': educ_w.value, 'ocupacion': ocup_w.value,
            'estado_civil': civil_w.value, 'problema_visual': prob_vis_w.value,
            'tipo_problema_visual': list(tipo_vis_w.value),
            'correccion_visual': correc_w.value, 'frecuencia_examen': examen_w.value,
            'trastorno_neurocog': t_cog_w.value, 'cambios_cognitivos': list(cambios_w.value),
            'trastorno_psiquiatrico': t_psiq_w.value, 'sintomas_psiquiatricos': sint_w.value,
            'tratamiento_psiquiatrico': trat_w.value,
            'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        print('✓ AdHoc guardado. Ejecute la Celda 5 → BCSE.')

btn_adhoc.on_click(on_adhoc)
display(btn_adhoc, out_adhoc)

---
## FASE 3 — BCSE
### Celda 5 — Guía de administración y registro de puntuaciones

> **Para el investigador:** Use el cuaderno de estímulos físico (Pearson). El participante responde verbalmente. Registre las puntuaciones directas aquí — el sistema calcula los puntajes ponderados automáticamente según el manual.

In [ ]:
display(HTML('<h2 style="font-family:Arial;color:#1a5276;">📊 BCSE — Registro de Puntuaciones</h2><div style="background:#eaf2ff;padding:12px;border-radius:6px;border-left:4px solid #1a5276;margin-bottom:15px;"><strong>Instrucción:</strong> Administre verbalmente con el cuaderno de estímulos físico (Pearson). Registre puntuaciones directas a continuación.</div>'))

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">1. Orientación (ítems 1-5, máx=5)</h3><p style="font-family:Arial;font-size:0.9em;">Pregunte: año actual, mes actual, día de la semana, día del mes, presidente del gobierno.</p>'))
orient_w = widgets.BoundedIntText(description='Puntuación directa (0-5):',
    min=0, max=5, value=0, layout=widgets.Layout(width='310px'), style={'description_width':'210px'})
display(orient_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">2. Estimación temporal (ítem 6)</h3><p style="font-family:Arial;font-size:0.9em;">Sin mirar el reloj, ¿qué hora es? Registre la diferencia en minutos entre la respuesta y la hora real.</p>'))
et_w = widgets.BoundedIntText(description='Diferencia en minutos:',
    min=0, max=720, value=0, layout=widgets.Layout(width='300px'), style={'description_width':'200px'})
display(et_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">3. Control mental (ítems 7-8)</h3><p style="font-family:Arial;font-size:0.9em;">Cuenta regresiva del 20 al 1 y meses del año al revés. Registre tiempo total y errores.</p>'))
cm_t_w = widgets.BoundedIntText(description='Tiempo total (segundos):',
    min=0, max=300, value=0, layout=widgets.Layout(width='290px'), style={'description_width':'200px'})
cm_e_w = widgets.BoundedIntText(description='Errores totales:',
    min=0, max=20,  value=0, layout=widgets.Layout(width='250px'), style={'description_width':'160px'})
display(cm_t_w, cm_e_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">4. Dibujo del reloj (ítem 9, máx=15)</h3><p style="font-family:Arial;font-size:0.9em;">Use la hoja en blanco del cuadernillo. Puntuar según criterios del manual (números + contorno + manecillas + centro).</p>'))
reloj_w = widgets.BoundedIntText(description='Puntuación directa (0-15):',
    min=0, max=15, value=0, layout=widgets.Layout(width='310px'), style={'description_width':'220px'})
display(reloj_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">5. Recuerdo incidental (ítem 10, máx=4)</h3><p style="font-family:Arial;font-size:0.9em;">¿Recuerda las 4 palabras de antes? Palabras objetivo: <strong>silla, gafas, pájaro, taza</strong>.</p>'))
recuerdo_w = widgets.BoundedIntText(description='Palabras recordadas (0-4):',
    min=0, max=4, value=0, layout=widgets.Layout(width='290px'), style={'description_width':'210px'})
display(recuerdo_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">6. Inhibición (ítem 11)</h3><p style="font-family:Arial;font-size:0.9em;">Nombrar solo las T (no las R). Registre tiempo, omisiones y comisiones.</p>'))
inh_t_w = widgets.BoundedIntText(description='Tiempo (segundos):',
    min=0, max=300, value=0, layout=widgets.Layout(width='280px'), style={'description_width':'180px'})
inh_o_w = widgets.BoundedIntText(description='Omisiones:',
    min=0, max=24,  value=0, layout=widgets.Layout(width='240px'), style={'description_width':'150px'})
inh_c_w = widgets.BoundedIntText(description='Comisiones:',
    min=0, max=24,  value=0, layout=widgets.Layout(width='240px'), style={'description_width':'150px'})
display(inh_t_w, inh_o_w, inh_c_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">7. Producción verbal (ítem 12)</h3><p style="font-family:Arial;font-size:0.9em;">Conceder 30 segundos. Registre el número de palabras producidas.</p>'))
pv_w = widgets.BoundedIntText(description='Palabras producidas:',
    min=0, max=50, value=0, layout=widgets.Layout(width='270px'), style={'description_width':'190px'})
display(pv_w)

btn_bcse = widgets.Button(description='Calcular y guardar BCSE',
    button_style='primary', layout=widgets.Layout(width='260px', height='42px'))
out_bcse = widgets.Output()

def pond_orient(d): return 0 if d<=3 else (5 if d==4 else 8)
def pond_et(m):     return 0 if m>55 else (1 if m>=31 else (2 if m>=19 else (3 if m>=10 else 4)))
def pond_cm_t(s):   return 0 if s>75 else (1 if s>=56 else (2 if s>=43 else (3 if s>=31 else 4)))
def pond_cm_e(e):   return 0 if e>3  else (2 if e>=2  else (4 if e==1  else 8))
def pond_reloj(d):  return 0 if d<=6 else (1 if d<=8  else (2 if d==9  else (3 if d<=11 else 4)))
def pond_recuerdo(d): return 0 if d==0 else (2 if d==1 else (4 if d==2 else 8))
def pond_inh_t(s):  return 0 if s>48 else (1 if s>=38 else (2 if s>=30 else (3 if s>=23 else 4)))
def pond_inh_o(o):  return 0 if o>=1 else 4
def pond_inh_c(c):
    for rango, pts in [(range(8,25),0),(range(7,8),1),(range(6,7),2),(range(5,6),3),
                       (range(4,5),4),(range(3,4),5),(range(2,3),6),(range(1,2),7),(range(0,1),8)]:
        if c in rango: return pts
    return 0
def pond_pv(p): return 0 if p<=6 else (1 if p==7 else (2 if p==8 else (4 if p==9 else 6)))

def on_bcse(b):
    with out_bcse:
        clear_output()
        p = [pond_orient(orient_w.value), pond_et(et_w.value),
             pond_cm_t(cm_t_w.value),     pond_cm_e(cm_e_w.value),
             pond_reloj(reloj_w.value),   pond_recuerdo(recuerdo_w.value),
             pond_inh_t(inh_t_w.value),   pond_inh_o(inh_o_w.value),
             pond_inh_c(inh_c_w.value),   pond_pv(pv_w.value)]
        total = sum(p)
        session_data['bcse'] = {
            'orientacion_directa': orient_w.value,  'orientacion_ponderada': p[0],
            'et_min': et_w.value,                   'et_ponderada': p[1],
            'cm_tiempo_seg': cm_t_w.value,          'cm_tiempo_ponderada': p[2],
            'cm_errores': cm_e_w.value,             'cm_errores_ponderada': p[3],
            'reloj_directa': reloj_w.value,         'reloj_ponderada': p[4],
            'recuerdo_directa': recuerdo_w.value,   'recuerdo_ponderada': p[5],
            'inhibicion_tiempo_seg': inh_t_w.value, 'inhibicion_tiempo_pond': p[6],
            'inhibicion_omisiones': inh_o_w.value,  'inhibicion_omis_pond': p[7],
            'inhibicion_comisiones': inh_c_w.value, 'inhibicion_comis_pond': p[8],
            'produccion_verbal': pv_w.value,        'produccion_verbal_pond': p[9],
            'total_bcse': total,
            'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        print(f'✓ BCSE calculado. Total: {total}/58')
        if total >= 47:   print('  → NORMAL')
        elif total >= 40: print('  → NORMAL-BAJO / LÍMITE')
        else:             print('  → BAJO / MUY BAJO (revisar criterio de exclusión)')
        print()
        print('Ejecute la Celda 6 → Prueba de Salud Visual.')

btn_bcse.on_click(on_bcse)
display(btn_bcse, out_bcse)

---
## FASE 4 — Prueba de Salud Visual
### Celda 6 — Agudeza visual, visión binocular e historia autoinformada

In [ ]:
display(HTML('<h2 style="font-family:Arial;color:#1a5276;border-bottom:2px solid #1a5276;padding-bottom:8px;">👁 PRUEBA DE SALUD VISUAL</h2>'))

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">Sección 1. Agudeza visual (cartilla Snellen a 3 m)</h3><p style="font-family:Arial;font-size:0.9em;">Evaluar un ojo a la vez cubriendo el otro. Registrar el nivel más alto alcanzado sin errores.</p>'))
oi_w  = widgets.Select(description='Ojo izquierdo:',
    options=['20/20','20/25','20/30','20/40','Peor de 20/40','No evaluable'],
    layout=widgets.Layout(width='340px'), style={'description_width':'130px'})
od_w  = widgets.Select(description='Ojo derecho:',
    options=['20/20','20/25','20/30','20/40','Peor de 20/40','No evaluable'],
    layout=widgets.Layout(width='340px'), style={'description_width':'130px'})
cor_w = widgets.RadioButtons(description='¿Usó corrección?',
    options=['Sí (gafas)','Sí (lentes de contacto)','No'],
    layout=widgets.Layout(width='360px'), style={'description_width':'140px'})
display(oi_w, od_w, cor_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">Sección 2. Visión binocular (5 imágenes anaglifas)</h3><p style="font-family:Arial;font-size:0.9em;">Presentar 5 imágenes anaglifas con gafas rojo/azul. Registrar cuántas identifica correctamente.</p>'))
ester_w = widgets.BoundedIntText(description='Ítems correctos (0-5):',
    min=0, max=5, value=0, layout=widgets.Layout(width='280px'), style={'description_width':'200px'})
display(ester_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">Sección 3. Historia visual autoinformada</h3>'))
preg_vis = [
    '¿Usa actualmente gafas o lentes de contacto?',
    '¿Le han diagnosticado miopía, astigmatismo, hipermetropía o estrabismo?',
    '¿Ha sido sometido a cirugía ocular (láser, cataratas, etc.)?',
    '¿Ha tenido visión doble o dificultad para enfocar en los últimos 6 meses?',
    '¿Tiene antecedentes de enfermedades neurológicas que afecten la visión?',
    '¿Le cuesta seguir objetos en movimiento o detectar diferencias de profundidad?'
]
hist_ws = []
for p in preg_vis:
    w = widgets.RadioButtons(description=p, options=['Sí','No','No sabe'],
        layout=widgets.Layout(width='700px'), style={'description_width':'500px'})
    hist_ws.append(w)
    display(w)

btn_vis = widgets.Button(description='Guardar y evaluar salud visual',
    button_style='primary', layout=widgets.Layout(width='290px', height='42px'))
out_vis = widgets.Output()

def cls_agudeza(v):
    return 'adecuada' if v in ['20/20','20/25'] else ('sospechosa' if v in ['20/30','20/40'] else 'limitada')

def on_vis(b):
    with out_vis:
        clear_output()
        ci = cls_agudeza(oi_w.value)
        cd = cls_agudeza(od_w.value)
        n_si      = sum(1 for w in hist_ws if w.value == 'Sí')
        excl_ag   = ci == 'limitada' and cd == 'limitada'
        excl_est  = ester_w.value < 3
        revision  = n_si >= 3
        session_data['salud_visual'] = {
            'ojo_izquierdo': oi_w.value,  'clase_oi': ci,
            'ojo_derecho':   od_w.value,  'clase_od': cd,
            'correccion_usada': cor_w.value,
            'estereopsia_correctos': ester_w.value,
            'historia': [w.value for w in hist_ws],
            'n_afirmativos': n_si,
            'exclusion_agudeza':    excl_ag,
            'exclusion_estereopsia': excl_est,
            'revision_adicional':   revision,
            'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        print(f'✓ Salud visual guardada.')
        print(f'  OI: {oi_w.value} ({ci}) | OD: {od_w.value} ({cd})')
        print(f'  Estereopsia: {ester_w.value}/5 | Historia: {n_si} afirmativos')
        print()
        if excl_ag:    print('  ⛔ EXCLUSIÓN: Agudeza visual limitada en ambos ojos.')
        elif excl_est: print('  ⛔ EXCLUSIÓN: Estereopsia insuficiente (< 3/5).')
        elif revision: print('  ⚠ Participación condicionada: revisar historia visual con experto.')
        else:          print('  ✓ Apto para las tareas experimentales.')
        print()
        print('Si el participante es apto, ejecute la Celda 7 → Guardar fases 1-4.')
        print('Luego continúe con la app Meta Quest 2 para la Fase 5.')

btn_vis.on_click(on_vis)
display(btn_vis, out_vis)

---
## Celda 7 — Guardado de fases 1-4 y transición a app Quest 2

> **Ejecutar después de completar la Fase 4 (salud visual).**  
> Una vez guardados los datos, continúe con la **app Meta Quest 2** para administrar la Fase 5 (190 tareas de percepción de profundidad).

In [ ]:
hora_fases_1_4 = datetime.datetime.now().strftime('%H:%M:%S')
session_data['hora_fin_fases_1_4'] = hora_fases_1_4

# Verificar que todas las fases están completas
fases_ok = {
    'Consentimiento': bool(session_data['consentimiento'].get('acepto')),
    'AdHoc':          bool(session_data['adhoc'].get('timestamp')),
    'BCSE':           bool(session_data['bcse'].get('total_bcse') is not None),
    'Salud visual':   bool(session_data['salud_visual'].get('timestamp')),
}

print('Verificación de fases completadas:')
for fase, ok in fases_ok.items():
    print(f'  {"✓" if ok else "✗"}  {fase}')
print()

if not all(fases_ok.values()):
    print('⚠ Hay fases incompletas. Complete todas las fases antes de guardar.')
else:
    # Guardar JSON de sesión (fases 1-4)
    ruta_sesion = os.path.join(DATA_DIR, f'{PARTICIPANT_ID}_session.json')
    with open(ruta_sesion, 'w', encoding='utf-8') as f:
        json.dump(session_data, f, ensure_ascii=False, indent=2)

    # Guardar CSV resumen de fases 1-4
    resumen = {
        'participant_id':    PARTICIPANT_ID,
        'es_piloto':         session_data['es_piloto'],
        'session_code':      SESSION_CODE,
        'fecha':             FECHA_SESION,
        'hora_inicio':       session_data['hora_inicio'],
        'hora_fin_fases_1_4': hora_fases_1_4,
        'edad':              session_data['adhoc'].get('edad'),
        'genero':            session_data['adhoc'].get('genero'),
        'nivel_educativo':   session_data['adhoc'].get('nivel_educativo'),
        'bcse_total':        session_data['bcse'].get('total_bcse'),
        'ojo_izquierdo':     session_data['salud_visual'].get('ojo_izquierdo'),
        'ojo_derecho':       session_data['salud_visual'].get('ojo_derecho'),
        'estereopsia':       session_data['salud_visual'].get('estereopsia_correctos'),
        'exclusion_agudeza': session_data['salud_visual'].get('exclusion_agudeza'),
        'exclusion_estereopsia': session_data['salud_visual'].get('exclusion_estereopsia'),
        'apto_para_tareas':  not (session_data['salud_visual'].get('exclusion_agudeza') or
                                  session_data['salud_visual'].get('exclusion_estereopsia')),
        'fase_5_medio':      'App nativa Meta Quest 2'
    }
    ruta_csv = os.path.join(RESULTS_DIR, f'{PARTICIPANT_ID}_fases_1_4.csv')
    pd.DataFrame([resumen]).to_csv(ruta_csv, index=False)

    print('='*60)
    print(f'FASES 1-4 COMPLETADAS — {PARTICIPANT_ID}')
    print('='*60)
    print(f'  Hora inicio:       {session_data["hora_inicio"]}')
    print(f'  Hora fin fases 1-4: {hora_fases_1_4}')
    print(f'  BCSE total:        {session_data["bcse"].get("total_bcse")}/58')
    print(f'  Salud visual:      OI={session_data["salud_visual"].get("ojo_izquierdo")} | '
          f'OD={session_data["salud_visual"].get("ojo_derecho")}')
    print(f'  Apto para tareas:  {resumen["apto_para_tareas"]}')
    if session_data.get('es_piloto'):
        print(f'  [Sesión piloto — datos excluidos del análisis estadístico]')
    print()
    print('Archivos guardados:')
    print(f'  → {ruta_sesion}')
    print(f'  → {ruta_csv}')
    print()
    print('─'*60)
    print('SIGUIENTE PASO → FASE 5: App Meta Quest 2')
    print('─'*60)
    print()
    print('1. Ponerse las gafas Meta Quest 2')
    print('2. Abrir la app "Percepción de Profundidad"')
    print(f'3. Ingresar el ID del participante: {PARTICIPANT_ID}')
    print('4. Completar las 190 tareas (duración estimada: 20-30 min)')
    print('5. Los datos se guardarán automáticamente en Drive al finalizar')